# Pseudobulk Analytics Testing (SAMPLE CORRECTED)

In [ ]:
import pseudobulk_analytics_utils_04 as pau
import scanpy as sc
import anndata as ad
import decoupler as dc
import gseapy as gp
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import glob
import os
from openpyxl.utils import get_column_letter
import seaborn as sns


%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import scProportionTest as pt


## 1. Load in file and keep transcripts of interest

### A. Load correct file

<div class="alert alert-block alert-info">
<b> Open h5ad, make sure correct file and confident in cell type annotations
</div>

In [ ]:
#### Read in file 

adata_query_X =ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_MTG_Finalized.h5ad")

In [ ]:
#### Make sure UMAP is same one as in 3b

sc.pl.umap(
    adata_query_X,
    color="predictions",
    frameon=False,
    legend_loc="on data",   # puts numbers on clusters
    legend_fontsize=10,
    legend_fontoutline=2
)

#### A1. Incorporate and filter based on MMSE

In [ ]:
# ── Load individual metadata to add MMSE ───────────────────────────────────────────────────
indiv_meta = pd.read_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_individual_metadata.csv')

# Check what's in it
print(indiv_meta.shape)
print(indiv_meta.columns.tolist())
print(indiv_meta.head(3))

In [ ]:
# ── Build lookup dictionaries ──────────────────────────────────────────────────
mmse_map = indiv_meta.set_index('individualID')['MMSE score']
casi_map  = indiv_meta.set_index('individualID')['CASI score']

# ── Map onto obs using individualID ───────────────────────────────────────────
adata_query_X.obs['MMSE score'] = adata_query_X.obs['individualID'].map(mmse_map)
adata_query_X.obs['CASI score'] = adata_query_X.obs['individualID'].map(casi_map)

# ── Convert to numeric ─────────────────────────────────────────────────────────
adata_query_X.obs['MMSE score'] = pd.to_numeric(adata_query_X.obs['MMSE score'], errors='coerce')
adata_query_X.obs['CASI score'] = pd.to_numeric(adata_query_X.obs['CASI score'], errors='coerce')

# ── Verify ─────────────────────────────────────────────────────────────────────
check = (adata_query_X.obs
    .drop_duplicates('individualID')
    [['individualID', 'comparison_group', 'MMSE score', 'CASI score']]
    .sort_values('comparison_group')
)
print(check.to_string())
print(f"\nMMSE missing donors: {check['MMSE score'].isna().sum()}")
print(f"CASI missing donors: {check['CASI score'].isna().sum()}")

In [ ]:
#### MMSE FILTER

adata_query_X = adata_query_X[
    (
        (adata_query_X.obs['comparison_group'] == 'AD')
    ) |
    (
        (adata_query_X.obs['comparison_group'] == 'PathologyControl') &
        (adata_query_X.obs['MMSE score'] > 23)
    )
].copy()

print(f"MMSE-filtered: {adata_query_X.n_obs} cells, {adata_query_X.obs['individualID'].nunique()} donors")
print(adata_query_X.obs.drop_duplicates('individualID')['comparison_group'].value_counts())

In [ ]:
##### AD DONOR SEPARATOIN BASED ON MMSE-linked CLUSTERS

# ── Define donor lists ─────────────────────────────────────────────────────────
high_mmse_donors = [
    'H20.33.004', 'H20.33.026', 'H20.33.029', 'H20.33.031', 'H21.33.005',
    'H21.33.007', 'H21.33.027', 'H21.33.029', 'H21.33.042'
]

low_mmse_donors = [
    'H20.33.020', 'H20.33.028', 'H20.33.037', 'H21.33.008', 'H21.33.009',
    'H21.33.039', 'H21.33.044', 'H21.33.046'
]

# ── Build comparison_group_2 ──────────────────────────────────────────────────
def assign_group2(row):
    if row['comparison_group'] == 'PathologyControl':
        return 'PathologyControl'
    elif row['individualID'] in high_mmse_donors:
        return 'high_MMSE'
    elif row['individualID'] in low_mmse_donors:
        return 'low_MMSE'
    else:
        return 'not_annotated'

adata_query_X.obs['comparison_group_2'] = adata_query_X.obs.apply(assign_group2, axis=1)

# ── Verify ─────────────────────────────────────────────────────────────────────
print(adata_query_X.obs['comparison_group_2'].value_counts())
print()
print(adata_query_X.obs.drop_duplicates('individualID')[
    ['individualID', 'comparison_group', 'comparison_group_2', 'MMSE score']
].sort_values('comparison_group_2').to_string())

In [ ]:
print(adata_query_X.obs['comparison_group_2'].unique())

In [ ]:
print(adata_query_X.obs.drop_duplicates('individualID')[['sex', 'comparison_group_2']].value_counts().sort_index())


In [ ]:
print(pd.crosstab(
    adata_query_X.obs.drop_duplicates('individualID')['Braak'],
    adata_query_X.obs.drop_duplicates('individualID')['comparison_group_2']
))

### B. Filter for important genes and keep high population cell types

In [ ]:
#### Get rid of all mitochondrial and unannotated gene transcripts

## Create mask
mask = ~(adata_query_X.var_names.str.startswith('MT-') | 
         adata_query_X.var_names.str.startswith('ENSG'))

## Filter
adata_query_X = adata_query_X[:, mask].copy()
print(f"Remaining genes: {adata_query_X.n_vars}")

In [ ]:
#### Get rid of cells types that have <3000 cells total

adata_query_X=pau.low_cell_type_remover(adata_query_X,n=3000)

In [ ]:
### Find diagnostic breakdown Severely Affected Donors

donor_breakdown = (adata_query_X.obs
    .drop_duplicates(subset='individualID')
    [['individualID', 'Severely Affected Donor', 'comparison_group']]
    .groupby(['Severely Affected Donor', 'comparison_group'])
    .size()
    .unstack(fill_value=0)
)
print(donor_breakdown)

### C. Run compositional analysis with scProportion test (takes ~10 hours, do only once!)

In [ ]:
group1 = 'PathologyControl'  # ← reference/denominator
group2 = 'low_MMSE'          # ← comparison/numerator

# Perform the permutation test
results = pt.permutation_test( adata_query_X,
                                group1,
                                group2,
                                group_col='comparison_group_2',
                                cell_type_col='predictions',
                                nperm=1000,
                                alpha=0.05,
                                n_bootstrap=1000,
                                verbose=True)

results

In [ ]:
results.to_csv("MTG_low_MMSE_proportion_test_results.csv")

plot = pt.point_range_plot(results, figsize=(6,4), fold_difference=0.5)
plt.savefig("MTG_low_MMSE_proportion_test_plot.png", dpi=600, bbox_inches="tight")

In [ ]:
# Check absolute cell counts per donor per cell type
abs_counts = (adata_query_X.obs
    .groupby(['individualID', 'comparison_group_2', 'predictions'])
    .size()
    .reset_index(name='n_cells')
)

# Mean absolute counts per group
for grp in ['low_MMSE', 'PathologyControl']:
    sub = abs_counts[abs_counts['comparison_group_2'] == grp]
    mean_counts = sub.groupby('predictions')['n_cells'].mean().sort_values(ascending=False)
    print(f"\n{grp} — mean cells per donor per cell type:")
    print(mean_counts.round(1).to_string())

## 2. Use "decoupler" for pseudobulking

<div class="alert alert-block alert-info">
<b> Use decoupler to pseudobulk the snRNAseq data. Sum counts by donor. Make sure to use raw counts
</div>

In [ ]:
pdata = dc.pp.pseudobulk(
    adata_query_X, 
    sample_col='individualID', 
    groups_col='predictions', # 🔴 Toggle between 'predictions' and 'full_leiden'
    layer='counts',    # Targets your raw data
    mode='sum'        # Sums the counts
)

print(pdata)

<div class="alert alert-block alert-info">
<b> pseudobulking changes meta data types, these must be changed to correct type again!! 
</div>

In [ ]:
print(pdata.obs[['sex', 'ageDeath_numeric', 'pmi', 'comparison_group_2', 'RIN', 'CPS']].dtypes)

In [ ]:
pdata.obs['ageDeath_numeric'] = pd.to_numeric(pdata.obs['ageDeath_numeric'], errors='coerce')
pdata.obs['pmi'] = pd.to_numeric(pdata.obs['pmi'], errors='coerce')
pdata.obs['RIN'] = pd.to_numeric(pdata.obs['RIN'], errors='coerce')
pdata.obs['CPS'] = pd.to_numeric(pdata.obs['CPS'], errors='coerce')
pdata.obs['sex'] = pdata.obs['sex'].astype('category')
pdata.obs['comparison_group_2'] = pdata.obs['comparison_group_2'].astype('category')

# Verify
print(pdata.obs[['sex', 'ageDeath_numeric', 'pmi', 'comparison_group_2', 'RIN', 'CPS']].dtypes)

In [ ]:
pdata_filtered = {}

for pred in pdata.obs['predictions'].unique():  
    adata_sub = pdata[pdata.obs['predictions'] == pred].copy() 
    
    # Perform filter 
    dc.pp.filter_by_expr(
        adata=adata_sub,
        group="comparison_group_2",
        min_count=10,
        min_total_count=15,
        large_n=10,
        min_prop=0.7,
    )
    dc.pp.filter_by_prop(
        adata=adata_sub,
        min_prop=0.1,
        min_smpls=2,
    )
    
    pdata_filtered[pred] = adata_sub

for pred, adata_sub in pdata_filtered.items():
    print(f"{pred}: {adata_sub.n_obs} donors, {adata_sub.n_vars} genes")


In [ ]:
os.makedirs("MTG_MMSE_low_PCA", exist_ok=True)

cell_types = list(pdata_filtered.keys())

ncols = 4
nrows = math.ceil(len(cell_types) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
axes = axes.flatten()

for idx, pred in enumerate(cell_types):
    ax = axes[idx]
    adata_sub = pdata_filtered[pred].copy()

    # -----------------------
    # BASIC QC TRANSFORM
    # -----------------------
    sc.pp.normalize_total(adata_sub)
    sc.pp.log1p(adata_sub)

    # -----------------------
    # PCA
    # -----------------------
    sc.pp.pca(adata_sub, n_comps=10)
    coords = adata_sub.obsm["X_pca"]

    # -----------------------
    # PLOT BY GROUP
    # -----------------------
    for group, color in [
        ("AD", "red"),
        ("PathologyControl", "blue")
    ]:
        mask = adata_sub.obs["comparison_group"] == group

        x = coords[mask, 0]
        y = coords[mask, 1]

        ax.scatter(
            x, y,
            c=color,
            label=group,
            alpha=0.8,
            s=70,
            edgecolor="black",
            linewidth=0.3
        )

    # -----------------------
    # OPTIONAL: LABEL DONORS (ONLY IF NEEDED)
    # -----------------------
    # Comment this out if cluttered
    for i, (x, y) in enumerate(coords[:, :2]):
        ax.text(
            x, y,
            adata_sub.obs["individualID"].iloc[i],
            fontsize=4,
            alpha=0.6
        )

    ax.set_title(pred, fontsize=9, fontweight="bold")
    ax.set_xlabel("PC1", fontsize=8)
    ax.set_ylabel("PC2", fontsize=8)
    ax.legend(fontsize=7)

# Hide unused subplots
for i in range(len(cell_types), len(axes)):
    axes[i].set_visible(False)

plt.suptitle("MTG_MMSE Pseudobulk PCA (Donor-Level)", fontsize=14, fontweight="bold")
plt.tight_layout()

plt.savefig(
    "pseudobulk_PCA_all_celltypes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved PCA plot: pseudobulk_PCA_all_celltypes.png")

## 3. DESeq2 for DEGs generation

In [ ]:
adata_query_X

In [ ]:
output_dir = "MTG_MMSE_pb_DEGs_low"

pau.DESeq2_pipeline(pdata_filtered, output_dir)

In [ ]:
deg_breakdown=pau.deg_calculator(output_dir)
deg_breakdown

## 4. GSEA

### 4A. Run GSEA

In [ ]:
#### Libraries to use 

#'GO_Biological_Process_2025'
#'MSigDB_Hallmark_2020'
#'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X'

#gp.get_library_name() # To get list of all available libraries 

In [ ]:
pau.gsea_pipeline(
    input_dir="MTG_MMSE_pb_DEGs_low", 
    output_dir="MTG_MMSE_pb_GSEA_low", 
    library="GO_Biological_Process_2025"
)

### 4B. Save excel files 

In [ ]:
import pandas as pd
import glob
import os
from openpyxl.utils import get_column_letter

In [ ]:
#### Configuration
input_folder = "/tscc/nfs/home/aopatel/pipeline/MTG_MMSE_pb_DEGs_low"  # Specify directory with DESeq2 results
file_pattern = "*_deseq2_results.csv" # Specify file pattern
output_name = "MTG_MMSE_low_DESeq2_results.xlsx" # Specify outputname 

#### Find and sort files
files = glob.glob(os.path.join(input_folder, file_pattern))
files.sort()

if not files:
    print(f"No DESeq2 files found in {input_folder}")
else:
    print(f"Found {len(files)} files. Compiling...")

    with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
        for file_path in files:
            # Extract cell type (e.g., 'Astrocyte')
            filename = os.path.basename(file_path)
            cell_type = filename.replace("_deseq2_results.csv", "")
            sheet_name = cell_type[:31]
            
            # Read CSV
            df = pd.read_csv(file_path)
            
            # Write to Excel
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Get worksheet for formatting
            worksheet = writer.sheets[sheet_name]
            
            # --- PROTECT GENE SYMBOLS ---
            # In DESeq2 CSVs, genes are almost always in the first column (Column A)
            # We force Column A to be Text to stop SEPT1 -> 1-Sep
            for cell in worksheet['A']:
                cell.number_format = '@'
            
            # Optional: Make the gene column a bit wider for readability
            worksheet.column_dimensions['A'].width = 20

    print(f"Successfully created: {output_name}")

In [ ]:
#### Configuration: Mapping file patterns to their final Excel output names
collections = {
    "GOBP": {
        "pattern": "*_GO_Biological_Process_2025_GSEA_significant.csv",
        "suffix": "_GO_Biological_Process_2025_GSEA_significant.csv",
        "output": "MTG_MMSE_GSEA_GOBP.xlsx"
    },
    "Hallmark": {
        "pattern": "*_MSigDB_Hallmark_2020_GSEA_significant.csv",
        "suffix": "_MSigDB_Hallmark_2020_GSEA_significant.csv",
        "output": "MTG_MMSE_GSEA_Hallmark.xlsx"
    },
    "Encode_ChEA": {
        "pattern": "*_ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X_GSEA_significant.csv",
        "suffix": "_ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X_GSEA_significant.csv",
        "output": "MTG_MMSE_GSEA_Encode_ChEA.xlsx"
    }
}



#### Specify directory with GSEA results
data_dir = "/tscc/nfs/home/aopatel/pipeline/MTG_MMSE_pb_GSEA_low/"

#### Main Loop: Process each category
for key, config in collections.items():
    # Find all matching files and sort them alphabetically
    search_path = os.path.join(data_dir, config["pattern"])
    files = glob.glob(search_path)
    if not files:
        print(f"Skipping {key}: No files found.")
        continue
    files.sort()
    
    print(f"Processing {key}...")
    
    with pd.ExcelWriter(config["output"], engine='openpyxl') as writer:
        for file_path in files:
            # Extract cell type for the tab name
            filename = os.path.basename(file_path)
            cell_type = filename.replace(config["suffix"], "")
            sheet_name = cell_type[:31]  # Excel limit
            
            # Read CSV
            df = pd.read_csv(file_path)
            
            # Write to Excel sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Get the openpyxl worksheet object for formatting
            worksheet = writer.sheets[sheet_name]
            
            # --- PROTECT GENE SYMBOLS FROM DATE CONVERSION ---
            if "leading_genes" in df.columns:
                # Find the column letter (e.g., 'G')
                col_idx = df.columns.get_loc("leading_genes") + 1
                col_letter = get_column_letter(col_idx)
                
                # Apply Text format ('@') to every cell in that column
                for cell in worksheet[col_letter]:
                    cell.number_format = '@'
                
                # Optional: Auto-adjust column width to 50 so you can see the genes
                worksheet.column_dimensions[col_letter].width = 50

    print(f"Successfully created: {config['output']}")

print("\nAll files are ready. Remember to double-click the .xlsx files directly!")

## 4C. ssGSEA

In [ ]:
all_scores = pau.ssgsea_pipeline_per_celltype(
    pdata_filtered,                          # same dict you fed to DESeq2_pipeline
    output_dir="MTG_ssGSEA_perCelltype",
    library="MSigDB_Hallmark_2020",          # same gmt you used for prerank
    donor_meta_cols=['comparison_group_2', 'MMSE score', 'CPS', 'sex', 'individualID']
)

In [ ]:
# How many cells underlie each donor's pseudobulk, per cell type
counts = (adata_query_X.obs
          .groupby(['predictions','individualID']).size()
          .reset_index(name='n_cells'))
# eyeball the low end for each cell type before trusting its ssGSEA spread
print(counts[counts['n_cells'] < 50].sort_values('n_cells').to_string())

In [ ]:
#### Uncover potential differential attributes (ECC)

import pandas as pd, numpy as np, glob, os
from scipy.stats import levene, spearmanr
from statsmodels.stats.multitest import multipletests

META = ['comparison_group_2','MMSE score','CASI score','CPS','sex','individualID']
ABUNDANT = ['L2/3 IT','L4 IT','L5 IT','L6 IT','L6 IT Car3',
            'Astrocyte','Oligodendrocyte','Microglia-PVM','Vip','Pvalb','Lamp5']

# map the cleaned filename back to a readable cell-type label
def ct_from_path(p):
    base = os.path.basename(p).split('_ENCODE')[0]
    return base.replace('-', '/').replace('_', ' ')  # L2-3_IT -> L2/3 IT (approx)

records = []
for f in glob.glob("MTG_ssGSEA_perCelltype/*_ssGSEA_perDonor.csv"):
    df = pd.read_csv(f, index_col=0)
    # recover a clean cell-type name from the file's own metadata if present,
    # else from filename
    ct = ct_from_path(f)
    tf_cols = [c for c in df.columns if c not in META]

    for tf in tf_cols:
        sub = df[['comparison_group_2', tf]].dropna()
        lm = sub[sub['comparison_group_2']=='low_MMSE'][tf]
        hm = sub[sub['comparison_group_2']=='high_MMSE'][tf]
        pc = sub[sub['comparison_group_2']=='PathologyControl'][tf]
        if len(lm) < 3 or len(pc) < 3:
            continue
        # Levene: does low_MMSE variance differ from PathologyControl?
        try:
            _, lev_p = levene(lm, pc)
        except Exception:
            lev_p = np.nan
        records.append({
            'cell_type': ct, 'TF': tf,
            'mean_low': lm.mean(), 'mean_high': hm.mean() if len(hm) else np.nan,
            'mean_PC': pc.mean(),
            'sd_low': lm.std(), 'sd_PC': pc.std(),
            'dose_ordered': (lm.mean() > hm.mean() > pc.mean()) if len(hm) else np.nan,
            'levene_p': lev_p,
        })

res = pd.DataFrame(records)

# ---- Summary 1: how often is low_MMSE MORE heterogeneous than PC? ----
ab = res[res['cell_type'].isin(ABUNDANT)].copy()
ab['low_more_variable'] = ab['sd_low'] > ab['sd_PC']
ab['levene_sig'] = ab['levene_p'] < 0.05

print("=== Heterogeneity summary (abundant cell types) ===")
print(f"TF×celltype tests: {len(ab)}")
print(f"low_MMSE sd > PC sd:        {ab['low_more_variable'].mean()*100:.0f}% of tests")
print(f"Levene significant (p<.05): {ab['levene_sig'].mean()*100:.0f}% of tests "
      f"(expect ~5% by chance)")
print(f"Dose-ordered low>high>PC:   {ab['dose_ordered'].mean()*100:.0f}% of tests")

# ---- Summary 2: per cell type, is low_MMSE coherent or heterogeneous? ----
print("\n=== Per-cell-type variance picture ===")
per_ct = ab.groupby('cell_type').agg(
    n_TF=('TF','size'),
    pct_low_more_variable=('low_more_variable', lambda x: 100*x.mean()),
    pct_levene_sig=('levene_sig', lambda x: 100*x.mean()),
    pct_dose_ordered=('dose_ordered', lambda x: 100*np.nanmean(x)),
).round(0)
print(per_ct.to_string())

# ---- Summary 3: the story TFs across abundant cell types ----
STORY_TFS = ['RELA ENCODE','FOS ENCODE','FOSL2 ENCODE','BCL3 ENCODE',
             'REST ENCODE','ESR1 CHEA','RFX5 ENCODE','PPARG CHEA','TP53 CHEA']
print("\n=== Story TFs: group means + Levene (abundant types) ===")
sel = ab[ab['TF'].isin(STORY_TFS)].copy()
for tf in STORY_TFS:
    block = sel[sel['TF']==tf]
    if block.empty: continue
    print(f"\n{tf}")
    for _, r in block.iterrows():
        flag = '  <-- low MORE variable' if r['low_more_variable'] and r['levene_sig'] else ''
        print(f"  {r['cell_type']:16} low={r['mean_low']:+.3f} high={r['mean_high']:+.3f} "
              f"PC={r['mean_PC']:+.3f} | sd_low={r['sd_low']:.3f} sd_PC={r['sd_PC']:.3f} "
              f"Lev_p={r['levene_p']:.2f}{flag}")

res.to_csv("ssGSEA_heterogeneity_summary_ECC.csv", index=False)
print("\nSaved full table -> ssGSEA_heterogeneity_summary.csv")

In [ ]:
#### Uncover potential differential attributes (Hallmark)

import pandas as pd, numpy as np, glob, os
from scipy.stats import levene
from statsmodels.stats.multitest import multipletests

META = ['comparison_group_2','MMSE score','CASI score','CPS','sex','individualID']
ABUNDANT = ['L2/3 IT','L4 IT','L5 IT','L6 IT','L6 IT Car3',
            'Astrocyte','Oligodendrocyte','Microglia-PVM','Vip','Pvalb','Lamp5']

LIB_TOKEN = '_MSigDB_Hallmark_2020'

def ct_from_path(p):
    base = os.path.basename(p).split(LIB_TOKEN)[0]
    return base.replace('-', '/').replace('_', ' ')

# ── Load only the Hallmark files ────────────────────────────────────────────────
files = glob.glob("MTG_ssGSEA_perCelltype/*Hallmark*_ssGSEA_perDonor.csv")
print(f"Found {len(files)} Hallmark files")
assert len(files) > 0, "No files matched — check the glob path"

records = []
for f in files:
    df = pd.read_csv(f, index_col=0)
    ct = ct_from_path(f)
    pw_cols = [c for c in df.columns if c not in META]
    for pw in pw_cols:
        sub = df[['comparison_group_2', pw]].dropna()
        lm = sub[sub['comparison_group_2']=='low_MMSE'][pw]
        hm = sub[sub['comparison_group_2']=='high_MMSE'][pw]
        pc = sub[sub['comparison_group_2']=='PathologyControl'][pw]
        if len(lm) < 3 or len(pc) < 3:
            continue
        try:
            _, lev_p = levene(lm, pc)
        except Exception:
            lev_p = np.nan
        records.append({
            'cell_type': ct, 'pathway': pw,
            'mean_low': lm.mean(), 'mean_high': hm.mean() if len(hm) else np.nan,
            'mean_PC': pc.mean(),
            'sd_low': lm.std(), 'sd_PC': pc.std(),
            'dose_ordered': (lm.mean() > hm.mean() > pc.mean()) if len(hm) else np.nan,
            'dose_ordered_rev': (lm.mean() < hm.mean() < pc.mean()) if len(hm) else np.nan,
            'levene_p': lev_p,
        })

res = pd.DataFrame(records)
print(f"Total pathway×celltype records: {len(res)}")

# ---- Summary 1: heterogeneity overview ----
ab = res[res['cell_type'].isin(ABUNDANT)].copy()
ab['low_more_variable'] = ab['sd_low'] > ab['sd_PC']
ab['levene_sig'] = ab['levene_p'] < 0.05
print("\n=== Heterogeneity summary (abundant cell types) — HALLMARK ===")
print(f"pathway×celltype tests: {len(ab)}")
print(f"low_MMSE sd > PC sd:            {ab['low_more_variable'].mean()*100:.0f}% of tests")
print(f"Levene significant (p<.05):    {ab['levene_sig'].mean()*100:.0f}% of tests (expect ~5% by chance)")
print(f"Dose-ordered low>high>PC:       {ab['dose_ordered'].mean()*100:.0f}% of tests")
print(f"Dose-ordered low<high<PC (rev): {ab['dose_ordered_rev'].mean()*100:.0f}% of tests")

# ---- Summary 2: per cell type ----
print("\n=== Per-cell-type variance picture ===")
per_ct = ab.groupby('cell_type').agg(
    n_pathways=('pathway','size'),
    pct_low_more_variable=('low_more_variable', lambda x: 100*x.mean()),
    pct_levene_sig=('levene_sig', lambda x: 100*x.mean()),
    pct_dose_up=('dose_ordered', lambda x: 100*np.nanmean(x)),
    pct_dose_down=('dose_ordered_rev', lambda x: 100*np.nanmean(x)),
).round(0)
print(per_ct.to_string())

# ---- Summary 3: story pathways ----
STORY_PATHWAYS = [
    'Oxidative Phosphorylation', 'Hypoxia',
    'TNF-alpha Signaling via NF-kB', 'Inflammatory Response',
    'Complement', 'Interferon Gamma Response', 'Interferon Alpha Response',
    'TGF-beta Signaling', 'PI3K/AKT/mTOR Signaling', 'mTORC1 Signaling',
    'Unfolded Protein Response', 'Reactive Oxygen Species Pathway',
    'Apoptosis', 'P53 Pathway',
]

all_pw = res['pathway'].unique()
def find_match(name):
    key = name.lower().replace(' ', '').replace('-', '').replace('/', '')
    for p in all_pw:
        if key in p.lower().replace(' ', '').replace('-', '').replace('/', ''):
            return p
    return None

print("\n=== Story pathways: group means + Levene (abundant types) ===")
for target in STORY_PATHWAYS:
    pw = find_match(target)
    if pw is None:
        print(f"\n{target}  [not found — check res['pathway'].unique()]")
        continue
    block = ab[ab['pathway']==pw]
    if block.empty:
        continue
    print(f"\n{pw}")
    for _, r in block.iterrows():
        flag = '  <-- low MORE variable' if r['low_more_variable'] and r['levene_sig'] else ''
        print(f"  {r['cell_type']:16} low={r['mean_low']:+.3f} high={r['mean_high']:+.3f} "
              f"PC={r['mean_PC']:+.3f} | sd_low={r['sd_low']:.3f} sd_PC={r['sd_PC']:.3f} "
              f"Lev_p={r['levene_p']:.2f}{flag}")

res.to_csv("ssGSEA_heterogeneity_summary_Hallmark.csv", index=False)
print("\nSaved -> ssGSEA_heterogeneity_summary_Hallmark.csv")

# Helpful if any pathway prints [not found]:
# print(sorted(res['pathway'].unique()))

In [3]:
### Assess donor heterogeneitys

import pandas as pd, numpy as np, glob, os
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

META = ['comparison_group_2','MMSE score','CASI score','CPS','sex','individualID']

# ── Choose which library to plot ────────────────────────────────────────────────
LIBRARY = 'ECC'   # 'Hallmark' or 'ECC'

if LIBRARY == 'Hallmark':
    PATTERN = "MTG_ssGSEA_perCelltype/*Hallmark*_ssGSEA_perDonor.csv"
    TOKEN = '_MSigDB_Hallmark_2020'
else:  # ECC
    PATTERN = "MTG_ssGSEA_perCelltype/*ENCODE_and_ChEA*_ssGSEA_perDonor.csv"
    TOKEN = '_ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X'

OUTDIR = f"MTG_ssGSEA_heatmaps_{LIBRARY}"
os.makedirs(OUTDIR, exist_ok=True)

# group colors for the column bar
GROUP_COLORS = {'low_MMSE':'#C0392B', 'high_MMSE':'#E67E22', 'PathologyControl':'#2E75B6'}
GROUP_ORDER  = ['PathologyControl', 'high_MMSE', 'low_MMSE']  # left → right

def ct_from_path(p):
    return os.path.basename(p).split(TOKEN)[0].replace('-', '/').replace('_', ' ')

files = glob.glob(PATTERN)
print(f"Found {len(files)} {LIBRARY} files")

for f in files:
    ct = ct_from_path(f)
    df = pd.read_csv(f, index_col=0)

    # split scores from metadata
    feat_cols = [c for c in df.columns if c not in META]
    scores = df[feat_cols].copy()              # donors × features
    meta = df[[c for c in META if c in df.columns]].copy()

    # ── order donors: by group, then by MMSE within group ───────────────────────
    meta['_grp_rank'] = meta['comparison_group_2'].map({g:i for i,g in enumerate(GROUP_ORDER)})
    donor_order = (meta.sort_values(['_grp_rank','MMSE score'])
                       .index.tolist())
    scores = scores.loc[donor_order]
    meta = meta.loc[donor_order]

    # ── matrix = features (rows) × donors (cols); z-score each feature across donors
    mat = scores.T                              # features × donors
    mat_z = mat.sub(mat.mean(axis=1), axis=0).div(mat.std(axis=1).replace(0,np.nan), axis=0)
    mat_z = mat_z.dropna(how='all')

    # ── column color bar by MMSE group ──────────────────────────────────────────
    col_colors = meta['comparison_group_2'].map(GROUP_COLORS)

    # ── plot ────────────────────────────────────────────────────────────────────
    n_feat = mat_z.shape[0]; n_don = mat_z.shape[1]
    g = sns.clustermap(
        mat_z,
        col_cluster=False,                      # keep donors in MMSE order
        row_cluster=True,                       # cluster gene sets by pattern
        col_colors=col_colors,
        cmap='RdBu_r', center=0, vmin=-2.5, vmax=2.5,
        figsize=(max(10, n_don*0.22 + 3), max(8, n_feat*0.28 + 2)),
        xticklabels=True, yticklabels=True,
        cbar_kws={'label': 'z-score (across donors)'},
        linewidths=0, dendrogram_ratio=(0.06, 0.10),
        colors_ratio=0.02,
        cbar_pos=(0.02, 0.83, 0.03, 0.13),      # ← top-left, out of the way
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7, rotation=0)
    g.ax_heatmap.set_xlabel('Donors (grouped: PathologyControl → high-MMSE → low-MMSE)', fontsize=9)
    g.ax_heatmap.set_ylabel('')

    # legend for the group bar — top-right so it doesn't collide with the colorbar
    handles = [mpatches.Patch(color=c, label=l) for l,c in GROUP_COLORS.items()]
    g.ax_col_dendrogram.legend(handles=handles, loc='upper right', ncol=1,
                               fontsize=8, frameon=False,
                               bbox_to_anchor=(1.0, 1.0), title=None)

    g.fig.suptitle(f"{ct} — {LIBRARY} ssGSEA (donors × gene sets, z-scored)",
                   fontsize=12, y=1.02)

    safe = ct.replace('/', '-').replace(' ', '_')
    g.savefig(f"{OUTDIR}/{safe}_{LIBRARY}_heatmap.png", dpi=200, bbox_inches='tight')
    plt.close(g.fig)
    print(f"  saved {ct}")

print(f"\nAll heatmaps in {OUTDIR}/")

Found 20 ECC files
  saved Pvalb
  saved OPC
  saved L4 IT
  saved L6 CT
  saved Lamp5
  saved L5 IT
  saved L6b
  saved Lamp5 Lhx6
  saved Chandelier
  saved Pax6
  saved L6 IT
  saved Sncg
  saved Vip
  saved L5/6 NP
  saved L6 IT Car3
  saved Oligodendrocyte
  saved Microglia/PVM
  saved L2/3 IT
  saved Astrocyte
  saved Sst

All heatmaps in MTG_ssGSEA_heatmaps_ECC/


## 5. CollecTRI

In [ ]:
# Load CollecTRI network
collectri = dc.op.collectri(organism="human")

# Output directory
os.makedirs("MTG_MMSE_low_CollecTRI", exist_ok=True)

# Run for all cell types
for filename in os.listdir("MTG_MMSE_pb_DEGs_low/"):
    if filename.endswith("_deseq2_results.csv"):
        cell_type = filename.replace("_deseq2_results.csv", "")
        print(f"--- Processing: {cell_type} ---")
        
        try:
            # Load DESeq2 results
            results_df = pd.read_csv(
                os.path.join("MTG_MMSE_pb_DEGs_low", filename), 
                index_col=0
            )
            
            # Prepare data matrix
            data = results_df[["stat"]].T.rename(
                index={"stat": "disease.vs.normal"}
            )
            
            # Run ULM
            tf_acts, tf_padj = dc.mt.ulm(data=data, net=collectri)
            
            # Filter significant TFs
            msk = (tf_padj.T < 0.05).iloc[:, 0]
            tf_acts_sig = tf_acts.loc[:, msk]
            
            print(f"Significant TFs: {tf_acts_sig.shape[1]}")
            
            # Save results
            tf_acts_sig.to_csv(
                os.path.join("MTG_MMSE_low_CollecTRI", 
                f"{cell_type}_collectri_results.csv")
            )
            
        except Exception as e:
            print(f"Error processing {cell_type}: {e}")

print("\nCollecTRI Pipeline Complete!")

In [ ]:
#### Save ALL CollecTRI results into excel

files = glob.glob("MTG_MMSE_low_CollecTRI/*_collectri_results.csv")
files.sort()

with pd.ExcelWriter("MTG_MMSE_low_CollecTRI_results.xlsx", engine='openpyxl') as writer:
    for file_path in files:
        cell_type = os.path.basename(file_path).replace("_collectri_results.csv", "")
        df = pd.read_csv(file_path, index_col=0).T
        df.to_excel(writer, sheet_name=cell_type[:31])
        
print("Saved: MTG_MMSE_low_CollecTRI_results.xlsx")

In [ ]:
#### Prepare CollecTRI heatmaps

os.makedirs("MTG_MMSE_low_heatmaps", exist_ok=True)

# ── Load all CollecTRI results ─────────────────────────────────────────────────
all_data = []
for file_path in sorted(glob.glob("MTG_MMSE_low_CollecTRI/*_collectri_results.csv")):
    cluster = os.path.basename(file_path).replace("_collectri_results.csv", "")
    df = pd.read_csv(file_path, index_col=0)
    df_t = df.T.rename(columns={"disease.vs.normal": cluster})
    all_data.append(df_t)

matrix = pd.concat(all_data, axis=1)
matrix = matrix.fillna(0)

# ── Sort columns by dominant cell type prefix ──────────────────────────────────
# Columns are like "EN_L2_3_IT_1", "Oligo_0", "Micro_10" etc.
# Define class order for sorting
class_order = ["EN", "IN", "Astro", "Oligo", "OPC", "Micro"]

def get_class_rank(col):
    for i, cls in enumerate(class_order):
        if col.startswith(cls):
            return i
    return len(class_order)  # unknown classes go last

sorted_cols = sorted(matrix.columns, key=lambda x: (get_class_rank(x), x))
matrix = matrix[sorted_cols]

# ── Select top variable TFs ────────────────────────────────────────────────────
matrix["variance"] = matrix.var(axis=1)
matrix = matrix.nlargest(40, "variance").drop(columns="variance")

# ── Dynamically compute vertical separator lines ───────────────────────────────
class_boundaries = []
current_class = None
count = 0
class_counts = []
for col in matrix.columns:
    col_class = next((cls for cls in class_order if col.startswith(cls)), "Other")
    if col_class != current_class:
        if current_class is not None:
            class_counts.append(count)
        current_class = col_class
        count = 1
    else:
        count += 1
class_counts.append(count)

boundaries = np.cumsum(class_counts)[:-1]  # exclude last (no line needed at end)

# ── Plot ───────────────────────────────────────────────────────────────────────
vmax = min(abs(matrix.values).max(), 5)
fig, ax = plt.subplots(figsize=(22, 14))
sns.heatmap(
    matrix,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "TF Activity Score", "shrink": 0.6},
    xticklabels=True,
    yticklabels=True,
)

for b in boundaries:
    ax.axvline(b, color="black", linewidth=2)

ax.set_title("MTG_MMSE AD vs PathologyControl — CollecTRI TF Activity",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig("MTG_MMSE_low_heatmaps/MTG_MMSE_low_CollecTRI_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved: MTG_MMSE_low_CollecTRI_heatmap.png")

## 6. PROGENy

In [ ]:
progeny = dc.op.progeny(organism="human")

os.makedirs("MTG_MMSE_low_PROGENy", exist_ok=True)

# ── Run PROGENy for all cell types ─────────────────────────────────────────────
for filename in os.listdir("MTG_MMSE_pb_DEGs_low/"):
    if filename.endswith("_deseq2_results.csv"):
        cell_type = filename.replace("_deseq2_results.csv", "")
        print(f"--- Processing: {cell_type} ---")
        
        try:
            results_df = pd.read_csv(
                os.path.join("MTG_MMSE_pb_DEGs_low", filename),
                index_col=0
            )
            
            data = results_df[["stat"]].T.rename(
                index={"stat": "disease.vs.normal"}
            )
            
            pw_acts, pw_padj = dc.mt.ulm(data=data, net=progeny)
            
            msk = (pw_padj.T < 0.05).iloc[:, 0]
            pw_acts_sig = pw_acts.loc[:, msk]
            
            print(f"Significant pathways: {pw_acts_sig.shape[1]}")
            
            pw_acts_sig.to_csv(
                os.path.join("MTG_MMSE_low_PROGENy", f"{cell_type}_progeny_results.csv")
            )
            
        except Exception as e:
            print(f"Error processing {cell_type}: {e}")

print("\nPROGENy Pipeline Complete!")

# ── Save joint Excel ───────────────────────────────────────────────────────────
with pd.ExcelWriter("MTG_MMSE_low_PROGENy_results.xlsx") as writer:
    for file_path in sorted(glob.glob("MTG_MMSE_low_PROGENy/*_progeny_results.csv")):
        cell_type = os.path.basename(file_path).replace("_progeny_results.csv", "")
        df = pd.read_csv(file_path, index_col=0)
        df.to_excel(writer, sheet_name=cell_type)
print("Saved: MTG_MMSE_low_PROGENy_results.xlsx")

# ── Build heatmap ──────────────────────────────────────────────────────────────
all_data = []
for file_path in sorted(glob.glob("MTG_MMSE_low_PROGENy/*_progeny_results.csv")):
    cell_type = os.path.basename(file_path).replace("_progeny_results.csv", "")
    df = pd.read_csv(file_path, index_col=0)
    df_t = df.T.rename(columns={"disease.vs.normal": cell_type})
    all_data.append(df_t)

if all_data:
    matrix = pd.concat(all_data, axis=1).fillna(0)

    # ── Sort columns by class prefix ───────────────────────────────────────────
    class_order = ["EN", "IN", "Astro", "Oligo", "OPC", "Micro"]

    def get_class_rank(col):
        for i, cls in enumerate(class_order):
            if col.startswith(cls):
                return i
        return len(class_order)

    sorted_cols = sorted(matrix.columns, key=lambda x: (get_class_rank(x), x))
    matrix = matrix[sorted_cols]

    # ── Dynamically compute vertical separator lines ───────────────────────────
    current_class = None
    count = 0
    class_counts = []
    for col in matrix.columns:
        col_class = next((cls for cls in class_order if col.startswith(cls)), "Other")
        if col_class != current_class:
            if current_class is not None:
                class_counts.append(count)
            current_class = col_class
            count = 1
        else:
            count += 1
    class_counts.append(count)
    boundaries = np.cumsum(class_counts)[:-1]

    vmax = min(abs(matrix.values).max(), 5)
    fig, ax = plt.subplots(figsize=(18, 8))
    sns.heatmap(
        matrix,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Pathway Activity Score", "shrink": 0.6},
        xticklabels=True,
        yticklabels=True,
    )

    for b in boundaries:
        ax.axvline(b, color="black", linewidth=2)

    ax.set_title("MTG_MMSE AD vs PathologyControl — PROGENy Pathway Activity",
                fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    plt.savefig("MTG_MMSE_low_heatmaps/MTG_MMSE_low_PROGENy_heatmap.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved: MTG_MMSE_low_PROGENy_heatmap.png")
else:
    print("No significant PROGENy results found!")

In [ ]:
### ANALYSIS TESTING ZONE FOR PROGENy

In [ ]:
from collections import Counter
import pandas as pd
import os

def get_gene_mechanism(stat, weight):
    """Classify gene contribution mechanism"""
    if stat > 0 and weight > 0:
        return "Activator UP"      # Direct PI3K activator, upregulated in AD
    elif stat < 0 and weight < 0:
        return "Suppressor DOWN"   # PI3K suppressor, downregulated in AD (disinhibition)
    elif stat > 0 and weight < 0:
        return "Suppressor UP"     # PI3K suppressor, upregulated in AD (counterbalancing)
    else:
        return "Activator DOWN"    # PI3K activator, downregulated in AD (counterbalancing)

pathway = "Hypoxia"
print(f"=== {pathway} Pathway — Full Gene Contribution Analysis ===\n")

all_contributions = {}
all_gene_stats = {}  # store stat values per gene per cell type

for filename in sorted(os.listdir("MTG_MMSE_pb_DEGs_low")):
    if filename.endswith("_deseq2_results.csv"):
        cell_type = filename.replace("_deseq2_results.csv", "")
        
        results_df = pd.read_csv(
            os.path.join("MTG_MMSE_pb_DEGs", filename),
            index_col=0
        )
        data = results_df[["stat"]].T.rename(index={"stat": "disease.vs.normal"})
        
        pathway_targets = progeny[progeny['source'] == pathway]
        available_genes = [g for g in pathway_targets['target'] if g in data.columns]
        weights_filtered = pathway_targets.set_index('target').loc[available_genes]
        expression = data[available_genes]
        contributions = expression * weights_filtered['weight']
        sample_contributions = contributions.iloc[0].sort_values(ascending=False)
        all_contributions[cell_type] = sample_contributions
        
        # Store stat and weight for each gene
        for gene in available_genes:
            if gene not in all_gene_stats:
                all_gene_stats[gene] = {
                    'weight': weights_filtered.loc[gene, 'weight'],
                    'stats': {}
                }
            all_gene_stats[gene]['stats'][cell_type] = data[gene].values[0]

# ── Find consistent contributors ───────────────────────────────────────────────
top_drivers = {}
top_inhibitors = {}

for ct, contrib in all_contributions.items():
    top_drivers[ct] = contrib.head(10).index.tolist()
    top_inhibitors[ct] = contrib.tail(10).index.tolist()

driver_counts = Counter([g for genes in top_drivers.values() for g in genes])
inhib_counts = Counter([g for genes in top_inhibitors.values() for g in genes])

# ── Print with mechanism labels ────────────────────────────────────────────────
print("=== Genes consistently contributing POSITIVELY to PI3K across cell types ===")
print(f"{'Gene':<15} {'Count':>6} {'Weight':>10} {'Mechanism':<20} {'Mean Stat':>10}")
print("-" * 65)

for gene, count in driver_counts.most_common(20):
    weight = all_gene_stats[gene]['weight']
    mean_stat = sum(all_gene_stats[gene]['stats'].values()) / len(all_gene_stats[gene]['stats'])
    mechanism = get_gene_mechanism(mean_stat, weight)
    print(f"{gene:<15} {count:>6} {weight:>10.3f} {mechanism:<20} {mean_stat:>10.3f}")

print("\n=== Genes consistently contributing NEGATIVELY to PI3K across cell types ===")
print(f"{'Gene':<15} {'Count':>6} {'Weight':>10} {'Mechanism':<20} {'Mean Stat':>10}")
print("-" * 65)

for gene, count in inhib_counts.most_common(20):
    weight = all_gene_stats[gene]['weight']
    mean_stat = sum(all_gene_stats[gene]['stats'].values()) / len(all_gene_stats[gene]['stats'])
    mechanism = get_gene_mechanism(mean_stat, weight)
    print(f"{gene:<15} {count:>6} {weight:>10.3f} {mechanism:<20} {mean_stat:>10.3f}")

# ── Summary by mechanism ───────────────────────────────────────────────────────
print("\n=== Summary by mechanism ===")
mechanisms = {"Activator UP": [], "Suppressor DOWN": [], 
              "Suppressor UP": [], "Activator DOWN": []}

for gene, info in all_gene_stats.items():
    mean_stat = sum(info['stats'].values()) / len(info['stats'])
    mechanism = get_gene_mechanism(mean_stat, info['weight'])
    mechanisms[mechanism].append(gene)

for mech, genes in mechanisms.items():
    print(f"\n{mech} ({len(genes)} genes):")
    # Show top genes by count in drivers or inhibitors
    relevant = {g: driver_counts.get(g, 0) + inhib_counts.get(g, 0) 
                for g in genes if driver_counts.get(g, 0) + inhib_counts.get(g, 0) > 0}
    top = sorted(relevant.items(), key=lambda x: x[1], reverse=True)[:5]
    for gene, count in top:
        print(f"  {gene} (appears in {count} cell types)")